In [3]:
import pandas as pd

In [4]:
import json
# Leer
with open("orders.json", encoding="utf-8") as f:
    orders = json.load(f)

In [5]:
rows = []

for orden in orders:

  refund = orden.get("refund") or {}

  ship = orden.get("shipping_address") or {}

  for item in orden.get("items", []):

    rows.append({

      "order_id": orden.get("order_id"),

      "ts": orden.get("ts"),

      "customer_id": orden.get("customer_id"),

      "country": orden.get("country"),

      "payment_method": orden.get("payment_method"),

      "status": orden.get("status"),

      "sku": item.get("sku"),

      "item_name": item.get("name"),

      "quantity": item.get("quantity"),

      "unit_price": item.get("unit_price"),

      "line_total": item.get("line_total"),

      "subtotal": orden.get("subtotal"),

      "discount": orden.get("discount"),

      "shipping": orden.get("shipping"),

      "tax": orden.get("tax"),

      "total": orden.get("total"),

      "coupon_code": orden.get("coupon_code"),

      "city": ship.get("city"),

      "postal_code": ship.get("postal_code"),

      "is_first_purchase": orden.get("is_first_purchase"),

      "notes": orden.get("notes"),

      "refund_reason": refund.get("refund_reason"),

      "refund_amount": refund.get("refund_amount")

    })

In [6]:
df = pd.DataFrame(rows)

In [5]:
df.head(8)

,order_id,ts,customer_id,country,payment_method,status,sku,item_name,quantity,unit_price,...,shipping,tax,total,coupon_code,city,postal_code,is_first_purchase,notes,refund_reason,refund_amount
0,4f841f5d-18d7-48d5-8ce2-63e589e5f8f1,2025-01-01T09:00:00Z,1094,DE,card,completed,SKU-00126,Queso manchego,1,149.60,...,4.99,31.42,171.05,WELCOME10,Porto,12082,True,Entregar por la tarde.,None,NaN
1,85184d5e-0431-4a02-a0f8-aa46f34ae99a,2025-01-01T16:00:00Z,1000,FR,transfer,refunded,SKU-00288,Queso manchego,3,122.39,...,2.99,160.10,849.21,WELCOME10,Zaragoza,20189,False,None,customer_request,849.21
2,85184d5e-0431-4a02-a0f8-aa46f34ae99a,2025-01-01T16:00:00Z,1000,FR,transfer,refunded,SKU-00280,Paella kit,3,131.73,...,2.99,160.10,849.21,WELCOME10,Zaragoza,20189,False,None,customer_request,849.21
3,1c195b44-a172-4748-ad1c-4513df36bc85,2025-01-01T23:00:00Z,1058,DE,card,completed,SKU-00434,Aceite de oliva,2,23.86,...,2.99,10.02,60.73,None,Bilbao,46178,False,Entregar por la tarde.,None,NaN
4,7458d24e-dbea-4f86-a90d-c3db8fb1ab49,2025-01-02T06:00:00Z,1035,IT,cash,completed,SKU-00396,Jamón ibérico,1,133.95,...,0.00,28.13,162.08,None,València,33283,False,None,None,NaN
5,fa14e9c9-d625-42cf-af64-042ce906f6e6,2025-01-02T13:00:00Z,1098,ES,paypal,pending,SKU-00084,Churros mix,3,147.18,...,4.99,116.93,623.06,WELCOME10,Madrid,36290,False,Entregar por la tarde.,None,NaN
6,fa14e9c9-d625-42cf-af64-042ce906f6e6,2025-01-02T13:00:00Z,1098,ES,paypal,pending,SKU-00474,Pimentón de la Vera,2,57.64,...,4.99,116.93,623.06,WELCOME10,Madrid,36290,False,Entregar por la tarde.,None,NaN
7,90c8e07f-1f1a-47f2-8ed1-c945bcab9c89,2025-01-02T20:00:00Z,1006,ES,paypal,completed,SKU-00330,Churros mix,2,177.51,...,2.99,174.09,1006.08,None,Paris,37666,False,Mensaje para regalo:\n¡Feliz cumpleaños! 🎉,None,NaN


In [7]:
total_pedidos = df["order_id"].nunique()
print("Total de pedidos:", total_pedidos)

Total de pedidos: 25


In [ ]:
df_completados = df[df["status"] == "completed"]
print("Pedidos completados:", df_completados["order_id"].nunique())


Pedidos completados: 11


In [26]:
paises_unicos = df["country"].unique()
print("Países únicos:", paises_unicos)


Países únicos: ['DE' 'FR' 'IT' 'ES' 'PT']


In [ ]:
df_pedidos = df.drop_duplicates(subset="order_id")

df_completados = df_pedidos[df_pedidos["status"] == "completed"]

total_por_pais = df_completados.groupby("country")["total"].sum().round(2)

print("Total recaudado por país (solo pedidos completados):")
print(total_por_pais)


Total recaudado por país (solo pedidos completados):
country
DE     708.94
ES    1006.08
FR     631.11
IT     378.05
PT     788.19
Name: total, dtype: float64


In [27]:
clientes_nuevos = df[df["is_first_purchase"] == True]["order_id"].nunique()
clientes_recurrentes = df[df["is_first_purchase"] == False]["order_id"].nunique()

print("Pedidos de clientes nuevos:", clientes_nuevos)
print("Pedidos de clientes recurrentes:", clientes_recurrentes)


Pedidos de clientes nuevos: 13
Pedidos de clientes recurrentes: 12


In [9]:
productos_cantidad = df.groupby("item_name")["quantity"].sum()

producto_mas_vendido = productos_cantidad.idxmax()
cantidad_mas_vendida = productos_cantidad.max()

print("Producto más vendido:", producto_mas_vendido)
print("Cantidad total vendida:", cantidad_mas_vendida)


Producto más vendido: Pimentón de la Vera
Cantidad total vendida: 23


In [11]:
df_pedidos = df.drop_duplicates(subset="order_id")

promedio_impuestos = (
    df_pedidos.groupby("payment_method")["tax"]
    .mean()
    .round(2)
)

print("Promedio de impuestos por método de pago:")
print(promedio_impuestos)


Promedio de impuestos por método de pago:
payment_method
card         96.80
cash        102.13
paypal       81.42
transfer    109.62
Name: tax, dtype: float64


In [30]:
df_refund = df[df["refund_reason"].notnull()]

reembolsos_por_causa = df_refund.groupby("refund_reason")["order_id"].nunique()
print("Pedidos con reembolso por causa:")
print(reembolsos_por_causa)


Pedidos con reembolso por causa:
refund_reason
customer_request    2
defective           3
lost_in_transit     4
Name: order_id, dtype: int64


In [ ]:
df_cupones = df[df["coupon_code"].notnull()]
cupones_usados = df_cupones.groupby("coupon_code")["order_id"].nunique()

descuento_por_cupon = df_cupones.drop_duplicates(subset="order_id").groupby("coupon_code")["discount"].sum()

print("Cupones usados (cantidad de veces):")
print(cupones_usados)
print("\nTotal de descuento aplicado por cada cupón:")
print(descuento_por_cupon)


Cupones usados (cantidad de veces):
coupon_code
FREESHIP     5
SUMMER5      4
WELCOME10    5
Name: order_id, dtype: int64

Total de descuento aplicado por cada cupón:
coupon_code
FREESHIP       0.00
SUMMER5       20.00
WELCOME10    205.26
Name: discount, dtype: float64


In [ ]:
df['tags'] = [orden.get('tags', []) for orden in orders for _ in orden.get('items', [])]

vip_df = df[df['tags'].apply(lambda x: 'vip' in x)]

productos_counts = vip_df.groupby('item_name')['quantity'].sum().sort_values(ascending=False)
print("Productos más comprados:\n", productos_counts)

metodos_counts = vip_df.groupby('payment_method')['order_id'].nunique().sort_values(ascending=False)
print("\nMétodos de pago preferidos:\n", metodos_counts)

ticket_promedio = vip_df.groupby('order_id')['total'].first().mean()
print("\nTicket promedio:\n", round(ticket_promedio, 2))


Productos más comprados:
 item_name
Pimentón de la Vera    12
Té verde                8
Café molido             7
Churros mix             6
Queso manchego          5
Paella kit              4
Chocolate 70%           3
Name: quantity, dtype: int64

Métodos de pago preferidos:
 payment_method
card        4
paypal      2
transfer    2
Name: order_id, dtype: int64

Ticket promedio:
 698.83
